##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# File Search 빠른 시작

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/File_Search.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>


[File Search 도구](http://ai.google.dev/gemini-api/docs/file-search)를 사용하면 Gemini를 활용한 강력한 검색 증강 생성(RAG) 애플리케이션을 구축할 수 있습니다. 관리형 저장소에 문서를 업로드한 다음 모델 생성 중 도구로 활용하여 Gemini가 정확한 인용과 함께 특정 데이터를 기반으로 질문에 답변할 수 있게 합니다.

이 빠른 시작에서는 다음을 배울 수 있습니다:

*   File Search Store 만들기.
*   저장소에 문서 업로드하기.
*   `generate_content`에서 저장소를 도구로 사용하기.
*   생성 중 사용된 출처 인용하기.
*   사용자 정의 메타데이터를 사용한 검색 결과 필터링.
*   문서 및 저장소 관리.

File Search 요금제에 대한 정보(무료로 제공되는 항목 포함)는 [요금 정보](https://ai.google.dev/gemini-api/docs/file-search#pricing)를 참고하세요.

## 의존성 설치

먼저 Google Gen AI SDK를 설치합니다.


In [ ]:
# SDK 1.49 introduced File Search
%pip install -U -q 'google-genai>=1.49.0'

### 인증

**중요:** File Search API는 인증 및 접근에 API 키를 사용합니다. 업로드된 파일은 API 키의 클라우드 프로젝트와 연결됩니다. API 키를 사용하는 다른 Gemini API와 달리, API 키는 파일 저장소에 업로드한 모든 데이터에 대한 접근 권한도 부여하므로 API 키를 안전하게 보관하는 데 각별히 주의하세요. API 키 보안 모범 사례는 Google의 [문서](https://support.google.com/googleapi/answer/6310037)를 참조하세요.

#### API 키 설정

다음 셀을 실행하려면 API 키를 `GEMINI_API_KEY`라는 이름의 Colab 시크릿에 저장해야 합니다. API 키가 없거나 Colab 시크릿 생성 방법이 궁금하다면 [인증](./Authentication.ipynb) 예제를 참고하세요.

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

## 기본 파일 검색

이 섹션에서는 샘플 문서를 다운로드하고, File Search Store를 만들고, 이를 사용하여 질문에 답변합니다.

### File Search Store 만들기

문서를 저장할 새 File Search Store를 만듭니다.


In [ ]:
file_search_store = client.file_search_stores.create(
    config=types.CreateFileSearchStoreConfig(
        display_name='My File Search Store'
    )
)

print(f"Created store: {file_search_store.name}")

Created store: fileSearchStores/my-file-search-store-9fimh45e9q14


### 샘플 문서 다운로드

샘플 텍스트 파일로 Project Gutenberg에서 "A Survey of Modernist Poetry"를 다운로드합니다.

In [ ]:
!wget -q https://www.gutenberg.org/cache/epub/76401/pg76401.txt -O sample_poetry.txt
!head sample_poetry.txt

The Project Gutenberg eBook of A survey of modernist poetry
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.



### 저장소에 파일 업로드

텍스트 파일을 저장소에 직접 업로드합니다. 수집 과정에 일부 처리가 포함되므로 검색 전에 완료될 때까지 기다려야 합니다.

In [ ]:
import time

upload_op = client.file_search_stores.upload_to_file_search_store(
    file_search_store_name=file_search_store.name,
    file='sample_poetry.txt',
    config=types.UploadToFileSearchStoreConfig(
        display_name='A Survey of Modernist Poetry',
    )
)

print(f"Upload started: {upload_op.name}")


while not (upload_op := client.operations.get(upload_op)).done:
    time.sleep(1)
    print(".", end="")

print()
print("Processing complete.")

Upload started: fileSearchStores/my-file-search-store-9fimh45e9q14/upload/operations/a-survey-of-modernist-poetr-rd7al7895r59
...
Processing complete.


### 대안: File API에서 가져오기

이미 [File API](https://ai.google.dev/gemini-api/docs/files)에 문서를 업로드했다면 파일 저장소로 직접 가져올 수 있습니다. 사용자가 파일로 일부 작업(예: 요약 생성)을 수행했고 파일을 저장소에서 사용하도록 승인한 경우에 유용합니다.

In [ ]:
file_ref = client.files.upload(
    file='sample_poetry.txt',
    config=types.UploadFileConfig(
        display_name='A Survey of Modernist Poetry',
        mime_type='text/plain',
    )
)
print(f"Uploaded via File API: {file_ref.name}")

import_op = client.file_search_stores.import_file(
    file_search_store_name=file_search_store.name,
    file_name=file_ref.name,
)

print(f"File import started: {import_op.name}")

while not (import_op := client.operations.get(import_op)).done:
    time.sleep(1)
    print(".", end="")

print()
print("Processing complete.")

Uploaded via File API: files/li0pn1it912i
File import started: fileSearchStores/my-file-search-store-9fimh45e9q14/operations/li0pn1it912i-ucqsutd9emb1
....
Processing complete.


### File Search로 콘텐츠 생성

이제 `generate_content` 요청에서 `file_search` 도구를 사용합니다. Gemini는 업로드된 문서를 사용하여 질문에 답변합니다.


In [ ]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

response = client.models.generate_content(
    model=MODEL_ID,
    contents='What does the text say about E.E. Cummings?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(
            file_search=types.FileSearch(
                file_search_store_names=[file_search_store.name],
            )
        )]
    )
)

print(response.text)

The text discusses E.E. Cummings as a significant poet whose work demands a "more vigorous imaginative effort" from the reader than what is typically applied to poetry. His innovations are considered to have a real place in the normal course of poetry-writing, and his acceptance is suggested, if not for his own sake, then for his potential effect on the future reading of poetry of any age or style.

One of Cummings' earlier and simpler poems is chosen for analysis in the text, despite its potential to elicit hostility similar to his later work. This particular poem is noted for its suitability for analysis, as it addresses a subject matter that the "plain reader" often seeks in poetry. It also appears in Mr. Louis Untermeyer's "Anthology of Modern American Poetry" alongside poets who are more willing to cater to the plain reader's intelligence level.

The text further suggests that Cummings writes according to a carefully constructed poetic system, but he refrains from providing a crit

#### 추가 필드

`FileSearch` 도구는 도구 작동 방식을 구성하는 옵션인 `top_k`와 `metadata_filter`를 제공합니다.

`top_k`는 검색 도구에서 반환되어 생성 단계로 전달될 청크 수를 제어합니다. 이전과 동일한 예제이지만 답변 생성에 1개의 청크만 사용됩니다. 올바른 청크가 하나만 있다는 것을 알 때(`k=1`) 또는 청크가 더 많이 겹칠 것으로 예상하여 더 많은 컨텍스트를 포함하고 싶을 때(`top_k` 높게 설정) 이 제어가 유용합니다.

메타데이터 필터링은 다음 섹션에서 설명되며, 전체 사양은 [API 레퍼런스](https://ai.google.dev/api/caching#FileSearch)에서 확인할 수 있습니다.

In [ ]:
top_K = 1 # @param {"allow-input":true, isTemplate: true}

response = client.models.generate_content(
    model=MODEL_ID,
    contents='What does the text say about E.E. Cummings?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(
            file_search=types.FileSearch(
                file_search_store_names=[file_search_store.name],
                top_k=top_K,
            )
        )]
    )
)

print(response.text)

### 그라운딩 메타데이터 검사

응답에는 출처 문서에 대한 인용 및 참조가 포함된 `grounding_metadata`가 포함됩니다. 생성 컨텍스트에서 사용된 문서 청크는 `grounding_metadata.grounding_chunks`에서 확인할 수 있으며 다음과 같은 형식입니다.

```python
[
  GroundingChunk(
    retrieved_context=GroundingChunkRetrievedContext(
      text="""(이 청크에 포함된 텍스트 스니펫)""",
      title='(문서 제목)'
    )
  ), ...
]
```

In [ ]:
import textwrap

grounding = response.candidates[0].grounding_metadata

if grounding and grounding.grounding_chunks:
    print(f"Found {len(grounding.grounding_chunks)} grounding chunks.")
    for i, chunk in enumerate(grounding.grounding_chunks, start=1):
        print(f"\nChunk {i} source: {chunk.retrieved_context.title}")
        print("Chunk text:")
        print(textwrap.indent(chunk.retrieved_context.text[:150] + "...", "  "))
else:
    print("No grounding metadata found.")

Found 5 grounding chunks.

Chunk 1 source: A Survey of Modernist Poetry
Chunk text:
  alterations in his critical
  attitude. In the first place, he must admit that what is called our
  common intelligence is the mind in its least active ...

Chunk 2 source: A Survey of Modernist Poetry
Chunk text:
  unusually suitable for analysis,
  because it is on just the kind of subject that the plain reader
  looks for in poetry. It appears, moreover, in Mr. L...

Chunk 3 source: A Survey of Modernist Poetry
Chunk text:
  4th of July the eyes of mice and Niagara
   Falls) that my “poems” are competing. They are also competing with
   each other, with elephants and with El...

Chunk 4 source: A Survey of Modernist Poetry
Chunk text:
  Mr. Cummings or for any poet to stabilize a poem once and
  for all. Punctuation marks in Mr. Cummings’ poetry are the bolts and
  axels that make the p...

Chunk 5 source: A Survey of Modernist Poetry
Chunk text:
  and to
  put _dream_ in its more logical position,

또한 `grounding_metadata`에는 응답 텍스트에서 지원 문서로의 참조를 제공하는 `grounding_supports`가 포함되어 있어 주석 제공에 활용할 수 있습니다.

지원 항목은 다음과 같은 형식입니다.

```python
[
  GroundingSupport(
    grounding_chunk_indices=[
      0,  # 이 항목이 대응하는 `grounding_chunks`의 인덱스
    ],
    segment=Segment(
      start_index=123,  # 생성된 텍스트로의 인덱스
      end_index=456,
      text='(지원되는 생성 텍스트 범위)'
    )
  ), ...
]
```

In [ ]:
from IPython.display import Markdown, display

# Accumulate the response as it is annotated.
annotated_response_parts = []

if not grounding or not grounding.grounding_supports:
    print("No grounding metadata or supports found for annotation.")
else:
    cursor = 0
    for support in grounding.grounding_supports:
        # Add the text before the current support
        annotated_response_parts.append(response.text[cursor:support.segment.start_index])

        # Construct the superscript citation from chunk IDs
        chunk_ids = ', '.join(map(str, support.grounding_chunk_indices))
        citation = f"<sup>{chunk_ids}</sup>"

        # Append the formatted, cited, supported text
        annotated_response_parts.append(f"**{support.segment.text}**{citation}")

        cursor = support.segment.end_index

    # Append any remaining text after the last support
    annotated_response_parts.append(response.text[cursor:])

    final_annotated_response = "".join(annotated_response_parts)
    display(Markdown(final_annotated_response))


The text discusses E.E. Cummings as a significant poet whose work demands a "more vigorous imaginative effort" from the reader than what is typically applied to poetry. **His innovations are considered to have a real place in the normal course of poetry-writing, and his acceptance is suggested, if not for his own sake, then for his potential effect on the future reading of poetry of any age or style**<sup>0</sup>.

One of Cummings' earlier and simpler poems is chosen for analysis in the text, despite its potential to elicit hostility similar to his later work. This particular poem is noted for its suitability for analysis, as it addresses a subject matter that the "plain reader" often seeks in poetry. It also appears in Mr. **Louis Untermeyer's "Anthology of Modern American Poetry" alongside poets who are more willing to cater to the plain reader's intelligence level**<sup>0, 1</sup>.

The text further suggests that Cummings writes according to a carefully constructed poetic system, but he refrains from providing a critical key to his poems, except as a "semi-prefatorial confidence." This implies that as poems become more independent, the need or sense for a technical guide diminishes. **The increasing difficulty of poems would not necessarily hinder understanding but rather make the reader less separated from poetry by technique**<sup>2</sup>.

Regarding his use of punctuation, the text describes punctuation marks in Cummings' poetry as "bolts and axels" that make the poem a "methodic and fool-proof piece of machinery" that requires common sense rather than imagination for its operation. **The strong reaction against his typography highlights the difficulty in engaging a reader's common sense as much as their imagination**<sup>3</sup>.

The text concludes by suggesting that while future poems might not all be written "in the Cummings way," his poetry is important as a "sign of local irritation in the poetic body" rather than a model for a new tradition. **It emphasizes the need to highlight the differences between good and bad poems to the reading public, especially during a time of popular but superficial education**<sup>4</sup>.

## 메타데이터 필터링

문서를 추가하는 동안 사용자 정의 메타데이터를 파일에 첨부하고 이를 사용하여 검색 결과를 필터링할 수 있습니다.

### 메타데이터가 있는 파일 업로드

"이상한 나라의 앨리스"를 다운로드하고 장르 및 저자 정보와 함께 업로드합니다.

In [ ]:
!wget -q https://www.gutenberg.org/files/11/11-0.txt -O alice_in_wonderland.txt
!head alice_in_wonderland.txt

*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll


In [ ]:
upload_op = client.file_search_stores.upload_to_file_search_store(
    file_search_store_name=file_search_store.name,
    file='alice_in_wonderland.txt',
    config=types.UploadToFileSearchStoreConfig(
        display_name='Alice in Wonderland',
        custom_metadata=[
            types.CustomMetadata(key='genre', string_value='fiction'),
            types.CustomMetadata(key='author', string_value='Lewis Carroll'),
        ]
    )
)

while not (upload_op := client.operations.get(upload_op)).done:
    time.sleep(1)
    print(".", end="")

print()
print("Upload complete.")

..
Upload complete.


사용자 정의 메타데이터는 `string_value`, `numeric_value` 또는 `string_list_value` 유형으로 제공할 수 있습니다.

In [ ]:
types.CustomMetadata.model_fields.keys() - {'key'}

{'numeric_value', 'string_list_value', 'string_value'}

### 메타데이터 필터로 쿼리

이제 두 책 모두에 적용될 수 있는 질문을 하되, 필터를 사용하여 하나로만 제한합니다. 예를 들어 '여왕'에 대해 묻되 'fiction'으로 필터링합니다.


In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents='Who is the Queen?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(
            file_search=types.FileSearch(
                file_search_store_names=[file_search_store.name],
                metadata_filter='genre = "fiction"'
            )
        )]
    )
)

print(response.text)
print('-' * 80)

if grounding := response.candidates[0].grounding_metadata:
  unique_titles = {c.retrieved_context.title for c in grounding.grounding_chunks}
  print(len(grounding.grounding_chunks), "sources used from", *unique_titles)
else:
  print("No sources used.")

Based on the information available, "the Queen" refers to the Queen of Hearts from Lewis Carroll's *Alice in Wonderland*. She is depicted as a tyrannical ruler who frequently shouts "Off with his head!" or "Off with her head!". She is known for her severe demeanor and short temper. For instance, she becomes crimson with fury when Alice responds surprisingly to her questions. The Queen is part of a grand procession alongside the King of Hearts, with soldiers and courtiers accompanying them. She is often seen playing croquet and sentencing players to execution.
--------------------------------------------------------------------------------
5 sources used from Alice in Wonderland


복잡한 필터를 만드는 방법에 대한 자세한 내용은 [AIP-160](https://google.aip.dev/160) 사양을 참조하세요.

## 문서 관리

저장소 내의 개별 문서도 관리할 수 있습니다.

### 문서 목록 조회

저장소에 현재 있는 모든 문서를 나열합니다.


In [ ]:
print(f"Documents in {file_search_store.name}:")

for doc in client.file_search_stores.documents.list(parent=file_search_store.name):
    print(f"- {doc.display_name} ({doc.name})")

Documents in fileSearchStores/my-file-search-store-9fimh45e9q14:
- A Survey of Modernist Poetry (fileSearchStores/my-file-search-store-9fimh45e9q14/documents/a-survey-of-modernist-poetr-rd7al7895r59)
- li0pn1it912i (fileSearchStores/my-file-search-store-9fimh45e9q14/documents/li0pn1it912i-ucqsutd9emb1)


### 문서 가져오기

처리 상태나 메타데이터 등 특정 문서의 세부 정보를 가져옵니다.


In [ ]:
# Get a document by ID
doc_id = doc.name  # Or set a specific ID here.
sample_doc = client.file_search_stores.documents.get(name=doc_id)

if sample_doc:
    print(f"Document details for {sample_doc.display_name}:")
    print(f"  Name: {sample_doc.name}")
    print(f"  Custom Metadata: {sample_doc.custom_metadata}")

Document details for li0pn1it912i:
  Name: fileSearchStores/my-file-search-store-9fimh45e9q14/documents/li0pn1it912i-ucqsutd9emb1
  Custom Metadata: None


### 문서 삭제

전체 저장소를 삭제하지 않고 저장소에서 특정 문서를 제거합니다.


In [ ]:
# Delete a specific document.
doc_to_be_deleted = doc_id

client.file_search_stores.documents.delete(
    name=doc_to_be_deleted,
    config=types.DeleteDocumentConfig(
        # Set force to delete a non-empty document.
        force=True
    )
)
print(f"Deleted document: {doc_to_be_deleted}")

# Verify deletion
print("\nRemaining documents:")
for doc in client.file_search_stores.documents.list(parent=file_search_store.name):
    print(f"- {doc.display_name}")

Deleted document: fileSearchStores/my-file-search-store-9fimh45e9q14/documents/li0pn1it912i-ucqsutd9emb1

Remaining documents:
- A Survey of Modernist Poetry


## File Search Store 관리

File Search Store를 나열, 조회, 삭제할 수 있습니다.


In [ ]:
print("Stores:")

for store in client.file_search_stores.list():
    print(f"- {store.name} ({store.display_name})")

Listing stores:
- fileSearchStores/my-file-search-store-9fimh45e9q14 (My File Search Store)


## 정리

불필요한 스토리지 비용이 발생하지 않도록 작업이 완료되면 File Search Store를 삭제하는 것이 좋습니다.


In [ ]:
client.file_search_stores.delete(name=file_search_store.name, config=types.DeleteFileSearchStoreConfig(force=True))
print(f"Deleted store: {file_search_store.name}")

## 다음 단계

File Search 도구에 대한 자세한 내용은 다음을 확인하세요.
 * [File Search 가이드](https://ai.google.dev/gemini-api/docs/file-search)
 * [요금 정보](https://ai.google.dev/gemini-api/docs/file-search#pricing)
 * 상세 [API 레퍼런스](https://ai.google.dev/api/file-search)
 * [File API 쿡북](./File_API.ipynb)